In [29]:
from pyspark.sql.functions import *

In [30]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("readCsvFile").getOrCreate()
df = spark.range(1000000).withColumn("id2",expr("id *2"))
df.show()

+---+---+
| id|id2|
+---+---+
|  0|  0|
|  1|  2|
|  2|  4|
|  3|  6|
|  4|  8|
|  5| 10|
|  6| 12|
|  7| 14|
|  8| 16|
|  9| 18|
| 10| 20|
| 11| 22|
| 12| 24|
| 13| 26|
| 14| 28|
| 15| 30|
| 16| 32|
| 17| 34|
| 18| 36|
| 19| 38|
+---+---+
only showing top 20 rows



In [31]:
df2 = df 
df2.show()

+---+---+
| id|id2|
+---+---+
|  0|  0|
|  1|  2|
|  2|  4|
|  3|  6|
|  4|  8|
|  5| 10|
|  6| 12|
|  7| 14|
|  8| 16|
|  9| 18|
| 10| 20|
| 11| 22|
| 12| 24|
| 13| 26|
| 14| 28|
| 15| 30|
| 16| 32|
| 17| 34|
| 18| 36|
| 19| 38|
+---+---+
only showing top 20 rows



In [32]:
joindf = df.join(df2,df.id==df2.id)
joindf.show()

26/04/01 16:26:58 WARN Column: Constructing trivially true equals predicate, 'id#205L = id#205L'. Perhaps you need to use aliases.


+---+---+---+---+
| id|id2| id|id2|
+---+---+---+---+
|  0|  0|  0|  0|
|  7| 14|  7| 14|
| 19| 38| 19| 38|
| 22| 44| 22| 44|
| 26| 52| 26| 52|
| 29| 58| 29| 58|
| 54|108| 54|108|
| 65|130| 65|130|
| 77|154| 77|154|
|112|224|112|224|
|113|226|113|226|
|130|260|130|260|
|155|310|155|310|
|167|334|167|334|
|191|382|191|382|
|196|392|196|392|
|198|396|198|396|
|222|444|222|444|
|237|474|237|474|
|241|482|241|482|
+---+---+---+---+
only showing top 20 rows



In [33]:
joindf.explain(True)

== Parsed Logical Plan ==
Join Inner, (id#205L = id#228L)
:- Project [id#205L, (id#205L * cast(2 as bigint)) AS id2#207L]
:  +- Range (0, 1000000, step=1, splits=Some(16))
+- Project [id#228L, (id#228L * cast(2 as bigint)) AS id2#229L]
   +- Range (0, 1000000, step=1, splits=Some(16))

== Analyzed Logical Plan ==
id: bigint, id2: bigint, id: bigint, id2: bigint
Join Inner, (id#205L = id#228L)
:- Project [id#205L, (id#205L * cast(2 as bigint)) AS id2#207L]
:  +- Range (0, 1000000, step=1, splits=Some(16))
+- Project [id#228L, (id#228L * cast(2 as bigint)) AS id2#229L]
   +- Range (0, 1000000, step=1, splits=Some(16))

== Optimized Logical Plan ==
Join Inner, (id#205L = id#228L)
:- Project [id#205L, (id#205L * 2) AS id2#207L]
:  +- Range (0, 1000000, step=1, splits=Some(16))
+- Project [id#228L, (id#228L * 2) AS id2#229L]
   +- Range (0, 1000000, step=1, splits=Some(16))

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [id#205L], [id#228L], Inner
   :- Sort [id#2

In [34]:
df.write.format("parquet").mode("overwrite").bucketBy(20,"id").saveAsTable("Bucketidse")

26/04/01 16:27:03 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/01 16:27:03 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/01 16:27:03 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/01 16:27:03 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/01 16:27:03 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/04/01 16:27:03 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/01 16:27:03 WARN MemoryManager: Total allocation exceeds 95.

In [35]:
bucketdf = spark.table("Bucketidse")

In [36]:
joindf2 = df.join(bucketdf, df.id == bucketdf.id)
joindf2.show()


+---+---+---+---+
| id|id2| id|id2|
+---+---+---+---+
|  0|  0|  0|  0|
|  1|  2|  1|  2|
|  2|  4|  2|  4|
|  3|  6|  3|  6|
|  4|  8|  4|  8|
|  5| 10|  5| 10|
|  6| 12|  6| 12|
|  7| 14|  7| 14|
|  8| 16|  8| 16|
|  9| 18|  9| 18|
| 10| 20| 10| 20|
| 11| 22| 11| 22|
| 12| 24| 12| 24|
| 13| 26| 13| 26|
| 14| 28| 14| 28|
| 15| 30| 15| 30|
| 16| 32| 16| 32|
| 17| 34| 17| 34|
| 18| 36| 18| 36|
| 19| 38| 19| 38|
+---+---+---+---+
only showing top 20 rows



In [37]:
joindf2.explain(True)

== Parsed Logical Plan ==
Join Inner, (id#205L = id#259L)
:- Project [id#205L, (id#205L * cast(2 as bigint)) AS id2#207L]
:  +- Range (0, 1000000, step=1, splits=Some(16))
+- SubqueryAlias spark_catalog.default.bucketidse
   +- Relation spark_catalog.default.bucketidse[id#259L,id2#260L] parquet

== Analyzed Logical Plan ==
id: bigint, id2: bigint, id: bigint, id2: bigint
Join Inner, (id#205L = id#259L)
:- Project [id#205L, (id#205L * cast(2 as bigint)) AS id2#207L]
:  +- Range (0, 1000000, step=1, splits=Some(16))
+- SubqueryAlias spark_catalog.default.bucketidse
   +- Relation spark_catalog.default.bucketidse[id#259L,id2#260L] parquet

== Optimized Logical Plan ==
Join Inner, (id#205L = id#259L)
:- Project [id#205L, (id#205L * 2) AS id2#207L]
:  +- Range (0, 1000000, step=1, splits=Some(16))
+- Filter isnotnull(id#259L)
   +- Relation spark_catalog.default.bucketidse[id#259L,id2#260L] parquet

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [id#205L], [id#

In [38]:
bucketdf2 = spark.table("Bucketidse")

In [39]:
joindf3 = bucketdf.join(bucketdf2, bucketdf.id == bucketdf2.id)
joindf3.show()

+------+-------+------+-------+
|    id|    id2|    id|    id2|
+------+-------+------+-------+
|500015|1000030|500015|1000030|
|500057|1000114|500057|1000114|
|500066|1000132|500066|1000132|
|500070|1000140|500070|1000140|
|500095|1000190|500095|1000190|
|500131|1000262|500131|1000262|
|500190|1000380|500190|1000380|
|500191|1000382|500191|1000382|
|500202|1000404|500202|1000404|
|500203|1000406|500203|1000406|
|500232|1000464|500232|1000464|
|500267|1000534|500267|1000534|
|500276|1000552|500276|1000552|
|500314|1000628|500314|1000628|
|500325|1000650|500325|1000650|
|500345|1000690|500345|1000690|
|500353|1000706|500353|1000706|
|500364|1000728|500364|1000728|
|500373|1000746|500373|1000746|
|500408|1000816|500408|1000816|
+------+-------+------+-------+
only showing top 20 rows



In [40]:
joindf3.explain(True)

== Parsed Logical Plan ==
Join Inner, (id#259L = id#291L)
:- SubqueryAlias spark_catalog.default.bucketidse
:  +- Relation spark_catalog.default.bucketidse[id#259L,id2#260L] parquet
+- SubqueryAlias spark_catalog.default.bucketidse
   +- Relation spark_catalog.default.bucketidse[id#291L,id2#292L] parquet

== Analyzed Logical Plan ==
id: bigint, id2: bigint, id: bigint, id2: bigint
Join Inner, (id#259L = id#291L)
:- SubqueryAlias spark_catalog.default.bucketidse
:  +- Relation spark_catalog.default.bucketidse[id#259L,id2#260L] parquet
+- SubqueryAlias spark_catalog.default.bucketidse
   +- Relation spark_catalog.default.bucketidse[id#291L,id2#292L] parquet

== Optimized Logical Plan ==
Join Inner, (id#259L = id#291L)
:- Filter isnotnull(id#259L)
:  +- Relation spark_catalog.default.bucketidse[id#259L,id2#260L] parquet
+- Filter isnotnull(id#291L)
   +- Relation spark_catalog.default.bucketidse[id#291L,id2#292L] parquet

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Broadcas